In [14]:
# ============================================================
# PHASE 1 — LEGAL BPE TOKENIZER V2
# ============================================================
#
# Purpose:
#   1. Train a robust BPE tokenizer on the legal Phase-1 corpus
#   2. Avoid the pair-counting bug
#   3. Preserve useful document/paragraph boundaries
#   4. Avoid unnecessary UNK tokens
#   5. Save vocab.json + merges.txt + tokenizer_config.json
#   6. Produce tokenized binary shards for Transformer training
#
# Designed for:
#   Decoder-only Transformer / Causal Language Model
#
# ============================================================

import os
import re
import json
import math
import time
import random
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)

PyTorch: 2.10.0+cpu
NumPy: 2.0.2


In [15]:
!pip install -q ijson
import os
import json
import random
import re
import unicodedata
import hashlib
from pathlib import Path
from collections import Counter
import ijson
import pandas as pd

SEED = 42
random.seed(SEED)

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

TEST_FILE = "Phase_1_Pretraining/Test/test.json"

OUTPUT_DIR = Path("/kaggle/working/phase1_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DIR = OUTPUT_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Ready.")

print("Configuration loaded.")
print("Output directory:", OUTPUT_DIR)

Ready.
Configuration loaded.
Output directory: /kaggle/working/phase1_data


In [16]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF")

os.environ["HF_TOKEN"] = HF_TOKEN

login(token=HF_TOKEN)

print("Hugging Face login successful.")

Hugging Face login successful.


In [17]:
# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(text: str) -> str:
    """
    Unicode normalization without destroying paragraph structure.

    We:
      - normalize Unicode using NFKC
      - normalize CRLF/CR -> LF
      - replace NUL characters
      - preserve newlines
      - preserve spaces
    """

    import unicodedata

    if not isinstance(text, str):
        text = str(text)

    text = unicodedata.normalize("NFKC", text)

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = text.replace("\x00", "")

    return text

In [18]:
# ============================================================
# PRE-TOKENIZATION — LEGAL CORPUS V3
# ============================================================

PRETOKEN_PATTERN = re.compile(
    r"""
    \n+
    |[ \t]*[\w]+(?:['’][\w]+)*
    |[ \t]*[^\w\s]
    |[ \t]+
    """,
    re.VERBOSE | re.UNICODE,
)


def pretokenize(text: str):
    text = normalize_text(text)
    return PRETOKEN_PATTERN.findall(text)


def is_word_like_piece(piece: str) -> bool:
    """
    True only for pieces containing at least one
    alphanumeric character.

    Examples:
        "court"       -> True
        " court"      -> True
        "12"          -> True
        "."           -> False
        " "           -> False
        "\\n"          -> False
    """
    return bool(
        piece.strip()
        and any(ch.isalnum() for ch in piece)
    )

In [19]:
from huggingface_hub import hf_hub_download

downloaded_files = {}

for source, filename in TRAIN_FILES.items():

    print(f"\nDownloading {source}...")

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    downloaded_files[source] = path

    print("Path:", path)

print("\nAll files downloaded.")


Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json

Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json

All files downloaded.


In [20]:
for source, path in downloaded_files.items():

    print("\n" + "=" * 70)
    print(source.upper())
    print("=" * 70)

    with open(path, "rb") as f:

        # Read ONLY the first JSON object
        first_record = next(
            ijson.items(f, "item")
        )

    print("Record type:", type(first_record))

    if isinstance(first_record, dict):

        print("Keys:", list(first_record.keys()))

        text = first_record.get("text", "")

        print("Text length:", len(text))

        print("\nFirst 500 characters:\n")
        print(text[:500])

[output removed before publishing: this cell printed raw dataset / test text]


In [21]:
from pathlib import Path

OUTPUT_ROOT = Path("/kaggle/working/legal_tokenizer_v3")

TOKENIZER_DIR = OUTPUT_ROOT / "tokenizer"
TOKENIZED_DIR = OUTPUT_ROOT / "tokenized"

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_ROOT)

Output directory: /kaggle/working/legal_tokenizer_v3


In [22]:
# ============================================================
# TOKENIZER CONFIGURATION
# ============================================================

VOCAB_SIZE = 32004

SPECIAL_TOKENS = [
    "<pad>",
    "<unk>",
    "<bos>",
    "<eos>",
]

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

BLOCK_SIZE = 2048

MIN_WORD_FREQ = 2

MAX_TRAIN_WORDS_PER_SOURCE = 100_000_000

SEED = 42

import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Vocabulary:", VOCAB_SIZE)
print("Block size:", BLOCK_SIZE)
print("Special tokens:", SPECIAL_TOKENS)

Vocabulary: 32004
Block size: 2048
Special tokens: ['<pad>', '<unk>', '<bos>', '<eos>']


In [23]:
import ijson


# ============================================================
# STREAMING LEGAL DOCUMENT READER
# ============================================================

def iter_json_documents(path):

    with open(path, "rb") as f:

        for record in ijson.items(f, "item"):

            if not isinstance(record, dict):
                continue

            text = record.get("text")

            if isinstance(text, str) and text:

                yield text

In [24]:
# ============================================================
# INSPECT DATA
# ============================================================

for source, path in downloaded_files.items():

    print("\n" + "=" * 80)
    print(source)
    print("=" * 80)

    try:

        doc = next(
            iter_json_documents(path)
        )

        print("Characters:", len(doc))
        print()
        print(doc[:1000])

    except Exception as e:

        print("ERROR:", repr(e))

[output removed before publishing: this cell printed raw dataset / test text]


In [25]:
# ============================================================
# PIECE FREQUENCIES
# ============================================================

def is_word_like_piece(piece):
    """
    Counts words/numbers toward the tokenizer training budget.

    Punctuation and whitespace do NOT count.
    """

    return bool(
        piece.strip()
        and any(ch.isalnum() for ch in piece)
    )


# ============================================================
# PIECE FREQUENCIES
# ============================================================

def build_piece_frequencies(
    path,
    max_words=None,
):
    """
    Stream a corpus and count pre-tokenized pieces.

    Also collects ALL characters before frequency filtering.
    This prevents rare characters from automatically becoming UNK.
    """

    piece_freq = Counter()
    char_freq = Counter()

    word_count = 0
    doc_count = 0

    for text in iter_json_documents(path):

        doc_count += 1

        pieces = pretokenize(text)

        for piece in pieces:

            piece_freq[piece] += 1

            # Build character alphabet from ALL data.
            char_freq.update(piece)

            if is_word_like_piece(piece):
                word_count += 1

            if (
                max_words is not None
                and word_count >= max_words
            ):
                break

        if (
            max_words is not None
            and word_count >= max_words
        ):
            break

    return (
        piece_freq,
        char_freq,
        doc_count,
        word_count,
    )

In [26]:
# ============================================================
# COLLECT TOKENIZER TRAINING STATISTICS
# ============================================================

all_piece_freq = Counter()
all_char_freq = Counter()

source_statistics = {}

for source_name, path in downloaded_files.items():

    print("\n" + "=" * 70)
    print("Scanning:", source_name)
    print("=" * 70)

    start = time.time()

    piece_freq, char_freq, docs, words = build_piece_frequencies(
        path,
        max_words=MAX_TRAIN_WORDS_PER_SOURCE,
    )

    # Merge statistics from this source
    all_piece_freq.update(piece_freq)
    all_char_freq.update(char_freq)

    source_statistics[source_name] = {
        "documents": docs,
        "words": words,
        "unique_pieces": len(piece_freq),
        "unique_characters": len(char_freq),
    }

    print(f"Documents       : {docs:,}")
    print(f"Words           : {words:,}")
    print(f"Unique pieces   : {len(piece_freq):,}")
    print(f"Unique characters: {len(char_freq):,}")
    print(f"Time            : {time.time() - start:.2f} sec")


print("\n" + "=" * 70)
print("COMBINED TOKENIZER STATISTICS")
print("=" * 70)

print(f"Total unique pieces    : {len(all_piece_freq):,}")
print(f"Total unique characters: {len(all_char_freq):,}")

print("\nMost frequent pieces:")
for piece, count in all_piece_freq.most_common(20):
    print(repr(piece), "->", count)

print("\nMost frequent characters:")
for char, count in all_char_freq.most_common(20):
    print(repr(char), "->", count)


Scanning: acts
Documents       : 15,491
Words           : 100,000,000
Unique pieces   : 418,995
Unique characters: 408
Time            : 348.10 sec

Scanning: high_courts
Documents       : 223,629
Words           : 100,000,000
Unique pieces   : 306,959
Unique characters: 256
Time            : 322.76 sec

Scanning: supreme_court
Documents       : 26,628
Words           : 100,000,000
Unique pieces   : 392,962
Unique characters: 302
Time            : 319.71 sec

Scanning: tribunals
Documents       : 57,692
Words           : 100,000,000
Unique pieces   : 502,332
Unique characters: 293
Time            : 332.86 sec

COMBINED TOKENIZER STATISTICS
Total unique pieces    : 1,155,386
Total unique characters: 562

Most frequent pieces:
'.' -> 35234698
' the' -> 31628954
' of' -> 19289919
',' -> 17910594
' to' -> 9255537
' and' -> 8049968
' in' -> 7747834
')' -> 6416437
'-' -> 5580949
'\n\n' -> 5366202
' for' -> 4614618
' is' -> 4401357
' be' -> 4198724
' that' -> 4098410
' by' -> 4092860
' a' ->

In [27]:
# ============================================================
# INITIAL VOCABULARY
# ============================================================

def build_initial_vocab(char_freq, special_tokens):
    """
    Build the initial BPE vocabulary.

    Order:
        1. Special tokens
        2. Every character observed in the corpus

    Important:
        The character alphabet is built from ALL observed
        characters, not only frequent words/pieces.
    """

    vocab = {}

    # --------------------------------------------------------
    # 1. Special tokens
    # --------------------------------------------------------
    for token in special_tokens:

        if token in vocab:
            raise ValueError(
                f"Duplicate special token: {token}"
            )

        vocab[token] = len(vocab)

    # --------------------------------------------------------
    # 2. Character vocabulary
    # --------------------------------------------------------
    for ch in char_freq.keys():

        if ch not in vocab:
            vocab[ch] = len(vocab)

    return vocab


vocab = build_initial_vocab(
    char_freq=all_char_freq,
    special_tokens=SPECIAL_TOKENS,
)


# ============================================================
# VALIDATION
# ============================================================

print("Initial vocabulary:", len(vocab))
print()

print("Special tokens:")
for token in SPECIAL_TOKENS:
    print(
        f"{token!r:10s} -> {vocab[token]}"
    )

print("\nNumber of characters:", len(all_char_freq))

print("\nFirst 30 characters:")
for token, idx in list(vocab.items())[len(SPECIAL_TOKENS):][:30]:
    print(repr(token), "->", idx)

Initial vocabulary: 566

Special tokens:
'<pad>'    -> 0
'<unk>'    -> 1
'<bos>'    -> 2
'<eos>'    -> 3

Number of characters: 562

First 30 characters:
'B' -> 4
'a' -> 5
'n' -> 6
'g' -> 7
'l' -> 8
'o' -> 9
'r' -> 10
'e' -> 11
' ' -> 12
'C' -> 13
'i' -> 14
't' -> 15
'y' -> 16
'P' -> 17
'A' -> 18
'Z' -> 19
'R' -> 20
'u' -> 21
's' -> 22
'(' -> 23
'm' -> 24
'd' -> 25
'V' -> 26
')' -> 27
'c' -> 28
',' -> 29
'1' -> 30
'9' -> 31
'6' -> 32
'K' -> 33


In [28]:
# ============================================================
# VALIDATE INITIAL VOCABULARY
# ============================================================

assert len(vocab) < VOCAB_SIZE, (
    f"Initial vocabulary ({len(vocab)}) >= "
    f"target vocabulary ({VOCAB_SIZE})"
)

# Every special token must have a unique ID
assert len(set(vocab[t] for t in SPECIAL_TOKENS)) == len(SPECIAL_TOKENS)

# IDs must be contiguous
assert set(vocab.values()) == set(range(len(vocab)))

print("Initial vocabulary size :", len(vocab))
print("Target vocabulary size  :", VOCAB_SIZE)
print("Available merge slots   :", VOCAB_SIZE - len(vocab))

print("\nSpecial token IDs:")
for token in SPECIAL_TOKENS:
    print(f"{token:>6} -> {vocab[token]}")

print("\n✓ Vocabulary validation passed")

Initial vocabulary size : 566
Target vocabulary size  : 32004
Available merge slots   : 31438

Special token IDs:
 <pad> -> 0
 <unk> -> 1
 <bos> -> 2
 <eos> -> 3

✓ Vocabulary validation passed


In [29]:
# ============================================================
# PREPARE BPE TRAINING PIECES
# ============================================================

training_piece_freq = Counter(
    {
        piece: freq
        for piece, freq in all_piece_freq.items()
        if freq >= MIN_WORD_FREQ
    }
)

print("Original unique pieces :", len(all_piece_freq))
print("Training pieces        :", len(training_piece_freq))
print("Removed rare pieces    :",
      len(all_piece_freq) - len(training_piece_freq))

print("\nMost frequent training pieces:")
for piece, freq in training_piece_freq.most_common(30):
    print(repr(piece), "->", freq)

Original unique pieces : 1155386
Training pieces        : 556285
Removed rare pieces    : 599101

Most frequent training pieces:
'.' -> 35234698
' the' -> 31628954
' of' -> 19289919
',' -> 17910594
' to' -> 9255537
' and' -> 8049968
' in' -> 7747834
')' -> 6416437
'-' -> 5580949
'\n\n' -> 5366202
' for' -> 4614618
' is' -> 4401357
' be' -> 4198724
' that' -> 4098410
' by' -> 4092860
' a' -> 3964085
' (' -> 3651262
' or' -> 3600197
'\n' -> 3318361
' as' -> 3115785
' on' -> 3063089
'(' -> 2703218
' The' -> 2399510
' not' -> 2380472
'/' -> 2269092
' was' -> 2242116
' shall' -> 2064249
' any' -> 1998726
' has' -> 1963608
' with' -> 1837863


In [30]:
# ============================================================
# INITIAL BPE SEQUENCES
# ============================================================

sequences = {
    piece: tuple(piece)
    for piece in training_piece_freq
}

print("Number of BPE sequences:", len(sequences))

# Sanity check
sample_piece = next(iter(sequences))

print("\nExample:")
print("Piece    :", repr(sample_piece))
print("Sequence :", sequences[sample_piece])

Number of BPE sequences: 556285

Example:
Piece    : 'Bangalore'
Sequence : ('B', 'a', 'n', 'g', 'a', 'l', 'o', 'r', 'e')


In [31]:
# ============================================================
# BPE PAIR COUNTING
# ============================================================

def count_pairs(sequences, piece_freq):
    """
    Count every occurrence of every adjacent pair.

    IMPORTANT:
    We count occurrences, NOT unique pairs.

    Example:
        banana

    contains:
        (b,a) -> 1
        (a,n) -> 2
        (n,a) -> 2
    """

    pair_counts = Counter()

    for piece, symbols in sequences.items():

        if len(symbols) < 2:
            continue

        freq = piece_freq[piece]

        for i in range(len(symbols) - 1):

            pair = (
                symbols[i],
                symbols[i + 1]
            )

            pair_counts[pair] += freq

    return pair_counts

In [32]:
# ============================================================
# TEST PAIR COUNTING
# ============================================================

test_sequences = {
    "banana": tuple("banana")
}

test_freq = {
    "banana": 1
}

test_counts = count_pairs(
    test_sequences,
    test_freq
)

print("Pair counts for 'banana':")

for pair, count in test_counts.items():
    print(pair, "->", count)

assert test_counts[("a", "n")] == 2
assert test_counts[("n", "a")] == 2

print("\n✓ Pair counting correctly counts repeated pairs")

Pair counts for 'banana':
('b', 'a') -> 1
('a', 'n') -> 2
('n', 'a') -> 2

✓ Pair counting correctly counts repeated pairs


In [33]:
# ============================================================
# TEST FREQUENCY WEIGHTING
# ============================================================

test_sequences = {
    "banana": tuple("banana")
}

test_freq = {
    "banana": 100
}

test_counts = count_pairs(
    test_sequences,
    test_freq
)

assert test_counts[("a", "n")] == 200
assert test_counts[("n", "a")] == 200
assert test_counts[("b", "a")] == 100

print("✓ Frequency weighting is correct")

✓ Frequency weighting is correct


In [34]:
# ============================================================
# BUILD BPE SEQUENCES
# ============================================================

# Each unique training piece is represented as a tuple of
# characters. Its frequency is stored separately in
# training_piece_freq.

sequences = {
    piece: tuple(piece)
    for piece in training_piece_freq
}

print("Number of sequences:", len(sequences))
print("Example:", next(iter(sequences.items())))

Number of sequences: 556285


In [35]:
# ============================================================
# INITIAL BPE PAIR STATISTICS
# ============================================================

pair_counts = count_pairs(
    sequences,
    training_piece_freq
)

print("Unique pairs:", len(pair_counts))

print("\nTop 50 pairs:")
for pair, count in pair_counts.most_common(50):

    print(
        f"{pair[0]!r:15} + "
        f"{pair[1]!r:15} "
        f"-> {count:,}"
    )

Unique pairs: 5542

Top 50 pairs:
' '             + 't'             -> 55,043,506
't'             + 'h'             -> 49,445,328
'h'             + 'e'             -> 44,765,285
' '             + 'a'             -> 36,615,333
' '             + 'o'             -> 33,090,839
'o'             + 'n'             -> 30,754,307
'e'             + 'r'             -> 30,165,911
'i'             + 'n'             -> 29,535,411
'a'             + 'n'             -> 25,100,365
'r'             + 'e'             -> 24,575,864
't'             + 'i'             -> 23,455,322
'e'             + 'd'             -> 21,521,860
'a'             + 't'             -> 21,226,357
'o'             + 'f'             -> 20,605,803
'e'             + 'n'             -> 20,464,876
' '             + 'i'             -> 20,323,398
'o'             + 'r'             -> 20,003,962
'n'             + 't'             -> 19,268,943
'n'             + 'd'             -> 18,366,697
'i'             + 'o'             -> 17,877,387
't'   

In [36]:
# ============================================================
# APPLY ONE BPE MERGE
# ============================================================

def merge_pair_in_sequence(
    symbols,
    best_pair,
    merged_token
):
    """
    Replace every occurrence of best_pair
    with merged_token.
    """

    new_symbols = []

    i = 0

    while i < len(symbols):

        if (
            i + 1 < len(symbols)
            and symbols[i] == best_pair[0]
            and symbols[i + 1] == best_pair[1]
        ):

            new_symbols.append(merged_token)

            i += 2

        else:

            new_symbols.append(symbols[i])

            i += 1

    return tuple(new_symbols)

In [37]:
# ============================================================
# TEST MERGE OPERATION
# ============================================================

symbols = tuple("banana")

merged = merge_pair_in_sequence(
    symbols,
    ("a", "n"),
    "an"
)

print("Original:", symbols)
print("Merged  :", merged)

assert merged == (
    "b",
    "an",
    "an",
    "a"
)

print("\n✓ Merge operation works")

Original: ('b', 'a', 'n', 'a', 'n', 'a')
Merged  : ('b', 'an', 'an', 'a')

✓ Merge operation works


In [38]:
# ============================================================
# BPE TRAINING
# ============================================================

def train_bpe(
    sequences,
    piece_freq,
    vocab,
    target_vocab_size,
    verbose_every=100,
):
    """
    Train BPE until target vocabulary size is reached.

    Returns:
        vocab
        merges
        final_sequences
    """

    merges = []

    target_merges = (
        target_vocab_size - len(vocab)
    )

    print("=" * 70)
    print("STARTING BPE TRAINING")
    print("=" * 70)

    print("Initial vocabulary :", len(vocab))
    print("Target vocabulary  :", target_vocab_size)
    print("Required merges    :", target_merges)

    start_time = time.time()

    for step in range(target_merges):

        # ----------------------------------------------------
        # Count all current pair occurrences
        # ----------------------------------------------------
        pair_counts = count_pairs(
            sequences,
            piece_freq
        )

        if not pair_counts:
            print("\nNo more mergeable pairs.")
            break

        # ----------------------------------------------------
        # Most frequent pair
        # ----------------------------------------------------
        best_pair, best_count = (
            pair_counts.most_common(1)[0]
        )

        merged_token = (
            best_pair[0] + best_pair[1]
        )

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------
        if merged_token in vocab:

            print(
                "\nWARNING: merged token already "
                "exists in vocabulary:"
            )

            print(repr(merged_token))

            # In this implementation we stop rather
            # than silently corrupting the merge list.
            break

        # ----------------------------------------------------
        # Add token to vocabulary
        # ----------------------------------------------------
        vocab[merged_token] = len(vocab)

        # ----------------------------------------------------
        # Record merge
        # ----------------------------------------------------
        merges.append(best_pair)

        # ----------------------------------------------------
        # Apply merge to every training sequence
        # ----------------------------------------------------
        new_sequences = {}

        for piece, symbols in sequences.items():

            new_sequences[piece] = (
                merge_pair_in_sequence(
                    symbols,
                    best_pair,
                    merged_token
                )
            )

        sequences = new_sequences

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------
        if (
            step == 0
            or (step + 1) % verbose_every == 0
            or step == target_merges - 1
        ):

            elapsed = time.time() - start_time

            print(
                f"Merge {step + 1:6,d}/"
                f"{target_merges:,} | "
                f"pair={best_pair!r} | "
                f"count={best_count:,} | "
                f"vocab={len(vocab):,} | "
                f"time={elapsed/60:.1f} min"
            )

    print("=" * 70)
    print("BPE TRAINING COMPLETE")
    print("=" * 70)

    print("Final vocabulary :", len(vocab))
    print("Total merges     :", len(merges))
    print(
        "Time             :",
        round((time.time() - start_time) / 60, 2),
        "minutes"
    )

    return vocab, merges, sequences

In [39]:
# ============================================================
# SMALL BPE TEST
# ============================================================

TEST_PIECES = Counter(
    {
        "banana": 100,
        "bandana": 80,
        "band": 60,
        "bank": 50,
        "court": 100,
        "courts": 70,
    }
)

test_sequences = {
    piece: tuple(piece)
    for piece in TEST_PIECES
}

test_vocab = {
    token: idx
    for idx, token in enumerate(
        SPECIAL_TOKENS
    )
}

for piece in sorted(
    set("".join(TEST_PIECES.keys()))
):

    if piece not in test_vocab:
        test_vocab[piece] = len(test_vocab)


test_vocab, test_merges, test_final_sequences = train_bpe(
    sequences=test_sequences,
    piece_freq=TEST_PIECES,
    vocab=test_vocab,
    target_vocab_size=len(test_vocab) + 20,
    verbose_every=5,
)

print("\nLearned merges:")
for i, merge in enumerate(test_merges):
    print(i, merge)

STARTING BPE TRAINING
Initial vocabulary : 15
Target vocabulary  : 35
Required merges    : 20
Merge      1/20 | pair=('a', 'n') | count=470 | vocab=16 | time=0.0 min
Merge      5/20 | pair=('co', 'u') | count=170 | vocab=20 | time=0.0 min
Merge     10/20 | pair=('band', 'ana') | count=80 | vocab=25 | time=0.0 min

No more mergeable pairs.
BPE TRAINING COMPLETE
Final vocabulary : 27
Total merges     : 12
Time             : 0.0 minutes

Learned merges:
0 ('a', 'n')
1 ('b', 'an')
2 ('an', 'a')
3 ('c', 'o')
4 ('co', 'u')
5 ('cou', 'r')
6 ('cour', 't')
7 ('ban', 'd')
8 ('ban', 'ana')
9 ('band', 'ana')
10 ('court', 's')
11 ('ban', 'k')


In [40]:
# ============================================================
# FINAL CHECKS BEFORE REAL BPE TRAINING
# ============================================================

print("=" * 70)
print("PRE-TRAINING CHECKS")
print("=" * 70)

print("Initial vocab size :", len(vocab))
print("Target vocab size  :", VOCAB_SIZE)
print("Training pieces    :", len(training_piece_freq))
print("Initial characters :", len(all_char_freq))

assert len(vocab) < VOCAB_SIZE
assert len(training_piece_freq) > 0
assert len(all_char_freq) > 0

# Special tokens exist
for token in SPECIAL_TOKENS:
    assert token in vocab

# IDs are contiguous
assert set(vocab.values()) == set(range(len(vocab)))

print("\n✓ All checks passed.")

PRE-TRAINING CHECKS
Initial vocab size : 566
Target vocab size  : 32004
Training pieces    : 556285
Initial characters : 562

✓ All checks passed.


In [41]:
# ============================================================
# OPTIMIZED INCREMENTAL BPE — INITIALIZE STATE
# ============================================================

import heapq
from collections import Counter, defaultdict


def initialize_incremental_bpe_state(
    sequences,
    piece_freq,
):
    """
    Build data structures for fast incremental BPE training.

    pair_counts:
        Global weighted frequency of every adjacent pair.

    pair_to_pieces:
        Which training pieces currently contain each pair.

    pair_versions:
        Version number used for lazy heap invalidation.

    heap:
        Max-heap implemented using negative counts.
    """

    pair_counts = Counter()
    pair_to_pieces = defaultdict(set)
    pair_versions = {}
    heap = []

    # --------------------------------------------------------
    # Scan each unique training piece ONCE
    # --------------------------------------------------------

    for piece, symbols in sequences.items():

        if len(symbols) < 2:
            continue

        freq = piece_freq[piece]
        seen_pairs = set()

        for i in range(len(symbols) - 1):

            pair = (
                symbols[i],
                symbols[i + 1],
            )

            # Weighted global count
            pair_counts[pair] += freq

            # We only need the piece once in this mapping
            seen_pairs.add(pair)

        for pair in seen_pairs:
            pair_to_pieces[pair].add(piece)

    # --------------------------------------------------------
    # Initialize heap
    # --------------------------------------------------------

    for pair, count in pair_counts.items():

        pair_versions[pair] = 0

        heap.append(
            (
                -count,
                pair,
                0,
            )
        )

    heapq.heapify(heap)

    return (
        pair_counts,
        pair_to_pieces,
        pair_versions,
        heap,
    )


print("Incremental BPE state initializer ready.")

Incremental BPE state initializer ready.


In [42]:
# ============================================================
# INCREMENTAL BPE — STATE UPDATE
# ============================================================

def update_pair_state_for_piece(
    piece,
    old_symbols,
    new_symbols,
    piece_frequency,
    pair_counts,
    pair_to_pieces,
    pair_versions,
    heap,
):
    """
    Update global BPE statistics for ONE changed training piece.
    """

    old_pair_counts = Counter(
        zip(
            old_symbols,
            old_symbols[1:],
        )
    )

    new_pair_counts = Counter(
        zip(
            new_symbols,
            new_symbols[1:],
        )
    )

    affected_pairs = (
        set(old_pair_counts)
        | set(new_pair_counts)
    )

    for pair in affected_pairs:

        old_count = old_pair_counts.get(
            pair,
            0,
        )

        new_count = new_pair_counts.get(
            pair,
            0,
        )

        if old_count == new_count:
            continue

        # ----------------------------------------------------
        # Update weighted global count
        # ----------------------------------------------------

        delta = (
            new_count - old_count
        ) * piece_frequency

        updated_total = (
            pair_counts.get(pair, 0)
            + delta
        )

        if updated_total < 0:
            raise RuntimeError(
                f"Negative pair count for {pair}: "
                f"{updated_total}"
            )

        # ----------------------------------------------------
        # Update piece membership
        # ----------------------------------------------------

        if old_count > 0 and new_count == 0:

            pair_to_pieces[pair].discard(
                piece
            )

        elif old_count == 0 and new_count > 0:

            pair_to_pieces[pair].add(
                piece
            )

        # ----------------------------------------------------
        # Update pair frequency
        # ----------------------------------------------------

        if updated_total > 0:

            pair_counts[pair] = (
                updated_total
            )

        else:

            pair_counts.pop(
                pair,
                None,
            )

        # ----------------------------------------------------
        # Version bump
        # ----------------------------------------------------

        version = (
            pair_versions.get(
                pair,
                -1,
            ) + 1
        )

        pair_versions[pair] = version

        # ----------------------------------------------------
        # Add fresh heap entry
        # ----------------------------------------------------

        if updated_total > 0:

            heapq.heappush(
                heap,
                (
                    -updated_total,
                    pair,
                    version,
                ),
            )


print("Incremental pair update ready.")

Incremental pair update ready.


In [43]:
# ============================================================
# INCREMENTAL BPE — SELECT BEST VALID PAIR
# ============================================================

def get_best_valid_pair(
    heap,
    pair_counts,
    pair_versions,
    vocab,
    blocked_pairs,
):
    """
    Return:

        (best_pair, count)

    while ignoring stale heap entries and invalid merges.
    """

    while heap:

        negative_count, pair, version = (
            heapq.heappop(heap)
        )

        current_count = -negative_count

        # ----------------------------------------------------
        # Pair was permanently blocked
        # ----------------------------------------------------

        if pair in blocked_pairs:
            continue

        # ----------------------------------------------------
        # Stale count
        # ----------------------------------------------------

        if pair_counts.get(pair, 0) != current_count:
            continue

        # ----------------------------------------------------
        # Stale heap version
        # ----------------------------------------------------

        if pair_versions.get(pair, -1) != version:
            continue

        # ----------------------------------------------------
        # Candidate merged token
        # ----------------------------------------------------

        merged_token = (
            pair[0] + pair[1]
        )

        # ----------------------------------------------------
        # Token already exists
        # ----------------------------------------------------

        if merged_token in vocab:

            blocked_pairs.add(pair)

            continue

        return (
            pair,
            current_count,
        )

    return None

In [44]:
# ============================================================
# TEST INCREMENTAL BPE STATE
# ============================================================

toy_piece_freq = Counter(
    {
        "banana": 100,
        "bandana": 80,
        "band": 60,
        "bank": 50,
        "court": 100,
        "courts": 70,
    }
)

toy_sequences = {
    piece: tuple(piece)
    for piece in toy_piece_freq
}

toy_vocab = {
    token: idx
    for idx, token in enumerate(
        SPECIAL_TOKENS
    )
}

for ch in sorted(
    set("".join(toy_piece_freq.keys()))
):

    if ch not in toy_vocab:
        toy_vocab[ch] = len(toy_vocab)


(
    toy_pair_counts,
    toy_pair_to_pieces,
    toy_pair_versions,
    toy_heap,
) = initialize_incremental_bpe_state(
    toy_sequences,
    toy_piece_freq,
)

toy_blocked_pairs = set()

# ------------------------------------------------------------
# Run several merges
# ------------------------------------------------------------

for test_step in range(20):

    # Fresh reference calculation
    reference_counts = count_pairs(
        toy_sequences,
        toy_piece_freq,
    )

    assert (
        toy_pair_counts == reference_counts
    ), (
        f"Pair-count mismatch before "
        f"test merge {test_step}"
    )

    candidate = get_best_valid_pair(
        toy_heap,
        toy_pair_counts,
        toy_pair_versions,
        toy_vocab,
        toy_blocked_pairs,
    )

    if candidate is None:
        break

    best_pair, best_count = candidate

    assert (
        best_count
        == reference_counts[best_pair]
    )

    merged_token = (
        best_pair[0]
        + best_pair[1]
    )

    affected_pieces = list(
        toy_pair_to_pieces[
            best_pair
        ]
    )

    # Add vocabulary token
    toy_vocab[merged_token] = (
        len(toy_vocab)
    )

    # Apply merge only to affected pieces
    for piece in affected_pieces:

        old_symbols = toy_sequences[piece]

        new_symbols = merge_pair_in_sequence(
            old_symbols,
            best_pair,
            merged_token,
        )

        update_pair_state_for_piece(
            piece=piece,
            old_symbols=old_symbols,
            new_symbols=new_symbols,
            piece_frequency=toy_piece_freq[piece],
            pair_counts=toy_pair_counts,
            pair_to_pieces=toy_pair_to_pieces,
            pair_versions=toy_pair_versions,
            heap=toy_heap,
        )

        toy_sequences[piece] = new_symbols


# Final comparison
reference_counts = count_pairs(
    toy_sequences,
    toy_piece_freq,
)

assert toy_pair_counts == reference_counts

print(
    f"✓ Incremental BPE state validated "
    f"for {test_step + 1} test merges"
)

✓ Incremental BPE state validated for 13 test merges


In [45]:
# ============================================================
# BUILD REAL INCREMENTAL BPE STATE
# ============================================================

print("=" * 70)
print("BUILDING REAL INCREMENTAL BPE STATE")
print("=" * 70)

state_start = time.time()

(
    pair_counts,
    pair_to_pieces,
    pair_versions,
    bpe_heap,
) = initialize_incremental_bpe_state(
    sequences,
    training_piece_freq,
)

blocked_pairs = set()

print(
    "Unique pairs        :",
    len(pair_counts),
)

print(
    "Pairs in heap       :",
    len(bpe_heap),
)

print(
    "Time                :",
    round(
        (time.time() - state_start) / 60,
        2,
    ),
    "minutes",
)

# Verify against the already computed naive statistics
assert pair_counts == count_pairs(
    sequences,
    training_piece_freq,
)

print("\n✓ Real BPE state matches reference pair counts.")

BUILDING REAL INCREMENTAL BPE STATE
Unique pairs        : 5542
Pairs in heap       : 5542
Time                : 0.07 minutes

✓ Real BPE state matches reference pair counts.


In [46]:
# ============================================================
# INCREMENTAL BPE TRAINER
# ============================================================

def save_incremental_bpe_checkpoint(
    path,
    vocab,
    merges,
    sequences,
):
    """
    Save the minimal state required for resume.

    Pair statistics are deliberately NOT saved.
    They can be reconstructed from sequences after restart.
    """

    checkpoint = {
        "vocab": vocab,
        "merges": merges,
        "sequences": sequences,
    }

    temp_path = str(path) + ".tmp"

    with open(
        temp_path,
        "wb",
    ) as f:

        pickle.dump(
            checkpoint,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    os.replace(
        temp_path,
        path,
    )


# ============================================================
# INCREMENTAL BPE TRAINER — CORRECTED
# ============================================================

def train_bpe_incremental(
    sequences,
    piece_freq,
    vocab,
    target_vocab_size,
    checkpoint_path=None,
    checkpoint_every=1000,
    verbose_every=100,
):
    """
    Fast incremental BPE trainer.

    Only sequences containing the selected pair are updated.
    """

    # --------------------------------------------------------
    # Initial state
    # --------------------------------------------------------

    initial_vocab_size = len(vocab)

    merges = []

    print("=" * 70)
    print("INCREMENTAL BPE TRAINING")
    print("=" * 70)

    print(
        "Initial vocabulary :",
        initial_vocab_size,
    )

    print(
        "Target vocabulary  :",
        target_vocab_size,
    )

    print(
        "Required merges    :",
        target_vocab_size - initial_vocab_size,
    )

    # --------------------------------------------------------
    # Build initial pair statistics
    # --------------------------------------------------------

    state_start = time.time()

    (
        pair_counts,
        pair_to_pieces,
        pair_versions,
        heap,
    ) = initialize_incremental_bpe_state(
        sequences,
        piece_freq,
    )

    blocked_pairs = set()

    print(
        f"Pair-state built in "
        f"{time.time() - state_start:.1f} sec"
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    training_start = time.time()

    while len(vocab) < target_vocab_size:

        candidate = get_best_valid_pair(
            heap,
            pair_counts,
            pair_versions,
            vocab,
            blocked_pairs,
        )

        if candidate is None:
            print("\nNo valid merge remains.")
            break

        best_pair, best_count = candidate

        merged_token = (
            best_pair[0] +
            best_pair[1]
        )

        # ----------------------------------------------------
        # Find only pieces containing selected pair
        # ----------------------------------------------------

        affected_pieces = list(
            pair_to_pieces.get(
                best_pair,
                set(),
            )
        )

        if not affected_pieces:
            raise RuntimeError(
                f"Selected pair has no affected pieces: "
                f"{best_pair}"
            )

        # ----------------------------------------------------
        # Add merged token
        # ----------------------------------------------------

        vocab[merged_token] = len(vocab)

        merges.append(best_pair)

        # ----------------------------------------------------
        # Update affected sequences only
        # ----------------------------------------------------

        for piece in affected_pieces:

            old_symbols = sequences[piece]

            new_symbols = merge_pair_in_sequence(
                old_symbols,
                best_pair,
                merged_token,
            )

            update_pair_state_for_piece(
                piece=piece,
                old_symbols=old_symbols,
                new_symbols=new_symbols,
                piece_frequency=piece_freq[piece],
                pair_counts=pair_counts,
                pair_to_pieces=pair_to_pieces,
                pair_versions=pair_versions,
                heap=heap,
            )

            sequences[piece] = new_symbols

        current_step = len(merges)

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            current_step == 1
            or current_step % verbose_every == 0
            or len(vocab) == target_vocab_size
        ):

            elapsed = (
                time.time() - training_start
            )

            rate = (
                current_step / elapsed * 60
                if elapsed > 0
                else 0
            )

            print(
                f"Merge {current_step:6,d}/"
                f"{target_vocab_size - initial_vocab_size:,}"
                f" | {best_pair!r} -> "
                f"{merged_token!r}"
                f" | count={best_count:,}"
                f" | affected={len(affected_pieces):,}"
                f" | vocab={len(vocab):,}"
                f" | rate={rate:.1f} merges/min"
            )

        # ----------------------------------------------------
        # Checkpoint
        # ----------------------------------------------------

        if (
            checkpoint_path is not None
            and current_step % checkpoint_every == 0
        ):

            save_incremental_bpe_checkpoint(
                checkpoint_path,
                vocab,
                merges,
                sequences,
            )

            print(
                f"  ✓ checkpoint saved "
                f"at merge {current_step:,}"
            )

    # --------------------------------------------------------
    # Final checkpoint
    # --------------------------------------------------------

    if checkpoint_path is not None:

        save_incremental_bpe_checkpoint(
            checkpoint_path,
            vocab,
            merges,
            sequences,
        )

    elapsed_minutes = (
        time.time() - training_start
    ) / 60

    print("\n" + "=" * 70)
    print("INCREMENTAL BPE TRAINING COMPLETE")
    print("=" * 70)

    print(
        "Final vocabulary :",
        len(vocab),
    )

    print(
        "Total merges     :",
        len(merges),
    )

    print(
        "Training time    :",
        round(elapsed_minutes, 2),
        "minutes",
    )

    return (
        vocab,
        merges,
        sequences,
    )

In [47]:
# ============================================================
# START REAL BPE TRAINING
# ============================================================
import pickle
BPE_CHECKPOINT_PATH = (
    TOKENIZER_DIR /
    "bpe_incremental_checkpoint.pkl"
)

print("=" * 70)
print("STARTING REAL LEGAL BPE TRAINING")
print("=" * 70)

print(
    "Initial vocabulary:",
    len(vocab),
)

print(
    "Target vocabulary:",
    VOCAB_SIZE,
)

print(
    "Merge operations:",
    VOCAB_SIZE - len(vocab),
)

print(
    "Checkpoint:",
    BPE_CHECKPOINT_PATH,
)

vocab, merges, sequences = train_bpe_incremental(
    sequences=sequences,
    piece_freq=training_piece_freq,
    vocab=vocab,
    target_vocab_size=VOCAB_SIZE,
    checkpoint_path=BPE_CHECKPOINT_PATH,
    checkpoint_every=1000,
    verbose_every=100,
)

STARTING REAL LEGAL BPE TRAINING
Initial vocabulary: 566
Target vocabulary: 32004
Merge operations: 31438
Checkpoint: /kaggle/working/legal_tokenizer_v3/tokenizer/bpe_incremental_checkpoint.pkl
INCREMENTAL BPE TRAINING
Initial vocabulary : 566
Target vocabulary  : 32004
Required merges    : 31438
Pair-state built in 4.2 sec
  ✓ checkpoint saved at merge 1,000
  ✓ checkpoint saved at merge 2,000
  ✓ checkpoint saved at merge 3,000
  ✓ checkpoint saved at merge 4,000
  ✓ checkpoint saved at merge 5,000
  ✓ checkpoint saved at merge 6,000
  ✓ checkpoint saved at merge 7,000
  ✓ checkpoint saved at merge 8,000
  ✓ checkpoint saved at merge 9,000
  ✓ checkpoint saved at merge 10,000
  ✓ checkpoint saved at merge 11,000
  ✓ checkpoint saved at merge 12,000
  ✓ checkpoint saved at merge 13,000
  ✓ checkpoint saved at merge 14,000
  ✓ checkpoint saved at merge 15,000
  ✓ checkpoint saved at merge 16,000
  ✓ checkpoint saved at merge 17,000
  ✓ checkpoint saved at merge 18,000
  ✓ checkpoint sa

In [48]:
print("Special tokens :", len(SPECIAL_TOKENS))
print("Characters     :", len(all_char_freq))
print("Initial vocab  :", len(SPECIAL_TOKENS) + len(all_char_freq))
print("BPE merges     :", len(merges))
print("Final vocab    :", len(vocab))

print(
    "\nCheck:",
    len(SPECIAL_TOKENS)
    + len(all_char_freq)
    + len(merges)
    == len(vocab)
)

Special tokens : 4
Characters     : 562
Initial vocab  : 566
BPE merges     : 31438
Final vocab    : 32004

Check: True


In [49]:
# ============================================================
# V3 PRE-TOKENIZATION CHECKS
# ============================================================

sample = "The court held the matter.\n\nSection 12 applies."

pieces = pretokenize(sample)

print("\nSample:")
print(repr(sample))

print("\nPretokenized pieces:")
for p in pieces:
    print(repr(p))

assert " " not in pieces, \
    "Standalone single-space token still exists."

assert all(
    p != "\t"
    for p in pieces
), "Standalone tab token found."

assert "The" in pieces
assert " court" in pieces
assert " held" in pieces

print("\n✓ V3 pretokenization checks passed.")

[output removed before publishing: this cell printed raw dataset / test text]


In [50]:
# ============================================================
# FINAL TOKENIZER PACKAGE
# Save everything required by the next tokenization notebook
# ============================================================

import json
import hashlib
import importlib.util
from pathlib import Path


# ============================================================
# 1. PATHS
# ============================================================

FINAL_TOKENIZER_DIR = (
    TOKENIZER_DIR / "final_tokenizer"
)

FINAL_TOKENIZER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

VOCAB_PATH = (
    FINAL_TOKENIZER_DIR / "vocab.json"
)

MERGES_PATH = (
    FINAL_TOKENIZER_DIR / "merges.txt"
)

CONFIG_PATH = (
    FINAL_TOKENIZER_DIR / "tokenizer_config.json"
)

PYTHON_PATH = (
    FINAL_TOKENIZER_DIR / "tokenizer.py"
)

MANIFEST_PATH = (
    FINAL_TOKENIZER_DIR / "manifest.json"
)


# ============================================================
# 2. VALIDATE CURRENT BPE STATE
# ============================================================

assert isinstance(vocab, dict)
assert isinstance(merges, list)

# Final vocabulary must be exactly the requested size.
assert len(vocab) == VOCAB_SIZE, (
    f"Vocabulary size is {len(vocab)}, "
    f"expected {VOCAB_SIZE}"
)

# Vocabulary IDs must be contiguous.
ids = sorted(
    int(idx)
    for idx in vocab.values()
)

assert ids == list(
    range(len(vocab))
), "Vocabulary IDs are not contiguous."


# ------------------------------------------------------------
# IMPORTANT:
#
# FINAL VOCAB =
#     special tokens
#   + character vocabulary
#   + BPE merges
#
# ------------------------------------------------------------

special_count = len(
    SPECIAL_TOKENS
)

character_count = len(
    all_char_freq
)

initial_vocab_from_stats = (
    special_count
    + character_count
)

initial_vocab_from_bpe = (
    len(vocab)
    - len(merges)
)

print("=" * 70)
print("BPE STATE VALIDATION")
print("=" * 70)

print(
    "Special tokens              :",
    special_count,
)

print(
    "Characters                  :",
    character_count,
)

print(
    "Initial vocab from stats    :",
    initial_vocab_from_stats,
)

print(
    "Initial vocab from BPE      :",
    initial_vocab_from_bpe,
)

print(
    "BPE merges                  :",
    len(merges),
)

print(
    "Final vocabulary            :",
    len(vocab),
)


# These MUST agree.
assert initial_vocab_from_bpe == (
    initial_vocab_from_stats
), (
    "Mismatch between the current "
    "vocab/merges and all_char_freq. "
    "This usually means stale BPE state."
)

expected_merges = (
    VOCAB_SIZE
    - initial_vocab_from_stats
)

assert len(merges) == (
    expected_merges
), (
    f"Unexpected merge count: "
    f"{len(merges)}; "
    f"expected {expected_merges}"
)

print(
    "\n✓ BPE vocabulary and merge history "
    "are internally consistent."
)


# ============================================================
# 3. SAVE VOCABULARY
# ============================================================

with open(
    VOCAB_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        vocab,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 4. SAVE MERGES
#
# IMPORTANT:
# Store every pair as a JSON array.
#
# This safely handles tokens containing:
#   - newline
#   - tab
#   - spaces
#   - Unicode characters
# ============================================================

with open(
    MERGES_PATH,
    "w",
    encoding="utf-8",
) as f:

    for a, b in merges:

        json.dump(
            [a, b],
            f,
            ensure_ascii=False,
        )

        f.write("\n")


# ============================================================
# 5. SAVE TOKENIZER CONFIGURATION
# ============================================================

tokenizer_config = {

    "tokenizer_type": "custom_bpe",

    "vocab_size": len(vocab),

    "block_size": BLOCK_SIZE,

    "special_tokens": {
        "pad_token": PAD_TOKEN,
        "unk_token": UNK_TOKEN,
        "bos_token": BOS_TOKEN,
        "eos_token": EOS_TOKEN,
    },

    "special_token_ids": {
        "pad_token_id": vocab[PAD_TOKEN],
        "unk_token_id": vocab[UNK_TOKEN],
        "bos_token_id": vocab[BOS_TOKEN],
        "eos_token_id": vocab[EOS_TOKEN],
    },

    "min_word_freq": MIN_WORD_FREQ,

    "normalization": "NFKC",

    "pretokenizer": (
        r"\n+|[ \t]+|"
        r"[\w]+(?:['’][\w]+)*|"
        r"[^\w\s]"
    ),

    "unicode": True,

    "training": {
        "vocab_size": VOCAB_SIZE,
        "num_merges": len(merges),
        "num_characters": character_count,
        "num_special_tokens": special_count,
        "initial_vocab_size": initial_vocab_from_stats,
    },
}


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        tokenizer_config,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 6. WRITE STANDALONE tokenizer.py
# ============================================================

tokenizer_code = r'''
import json
import re
import unicodedata


# ============================================================
# PRETOKENIZATION
# ============================================================

PRETOKEN_PATTERN = re.compile(
    r"""
    \n+
    |[ \t]*[\w]+(?:['’][\w]+)*
    |[ \t]*[^\w\s]
    |[ \t]+
    """,
    re.VERBOSE | re.UNICODE,
)


def normalize_text(text: str) -> str:

    if not isinstance(text, str):
        text = str(text)

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    text = text.replace(
        "\r\n",
        "\n",
    )

    text = text.replace(
        "\r",
        "\n",
    )

    text = text.replace(
        "\x00",
        "",
    )

    return text


def pretokenize(text: str):

    text = normalize_text(text)

    return PRETOKEN_PATTERN.findall(
        text
    )


# ============================================================
# BPE TOKENIZER
# ============================================================

class BPETokenizer:

    def __init__(
        self,
        vocab,
        merges,
        special_tokens=None,
    ):

        self.vocab = {
            str(token): int(idx)
            for token, idx in vocab.items()
        }

        self.id_to_token = {
            idx: token
            for token, idx in self.vocab.items()
        }

        self.merges = [
            tuple(pair)
            for pair in merges
        ]

        self.merge_ranks = {
            pair: rank
            for rank, pair in enumerate(
                self.merges
            )
        }

        self.vocab_size = len(
            self.vocab
        )

        if special_tokens is None:

            special_tokens = {
                "pad_token": "<pad>",
                "unk_token": "<unk>",
                "bos_token": "<bos>",
                "eos_token": "<eos>",
            }

        self.special_tokens = (
            special_tokens
        )

        self.pad_token = (
            special_tokens["pad_token"]
        )

        self.unk_token = (
            special_tokens["unk_token"]
        )

        self.bos_token = (
            special_tokens["bos_token"]
        )

        self.eos_token = (
            special_tokens["eos_token"]
        )

        self.pad_id = self.vocab.get(
            self.pad_token
        )

        self.unk_id = self.vocab.get(
            self.unk_token
        )

        self.bos_id = self.vocab.get(
            self.bos_token
        )

        self.eos_id = self.vocab.get(
            self.eos_token
        )

    # ========================================================
    # APPLY BPE
    # ========================================================

    def _apply_bpe(
        self,
        symbols,
    ):

        symbols = list(symbols)

        while len(symbols) >= 2:

            best_pair = None

            best_rank = float("inf")

            # Find the highest-priority merge
            # currently applicable.

            for i in range(
                len(symbols) - 1
            ):

                pair = (
                    symbols[i],
                    symbols[i + 1],
                )

                rank = self.merge_ranks.get(
                    pair
                )

                if (
                    rank is not None
                    and rank < best_rank
                ):

                    best_rank = rank

                    best_pair = pair

            if best_pair is None:
                break

            merged = []

            i = 0

            while i < len(symbols):

                if (
                    i < len(symbols) - 1
                    and symbols[i]
                    == best_pair[0]
                    and symbols[i + 1]
                    == best_pair[1]
                ):

                    merged.append(
                        symbols[i]
                        + symbols[i + 1]
                    )

                    i += 2

                else:

                    merged.append(
                        symbols[i]
                    )

                    i += 1

            symbols = merged

        return symbols

    # ========================================================
    # ENCODE
    # ========================================================

    def encode(
        self,
        text,
    ):

        if not text:
            return []

        text = normalize_text(
            text
        )

        ids = []

        pieces = pretokenize(
            text
        )

        for piece in pieces:

            symbols = list(
                piece
            )

            symbols = self._apply_bpe(
                symbols
            )

            for token in symbols:

                token_id = self.vocab.get(
                    token
                )

                if token_id is not None:

                    ids.append(
                        token_id
                    )

                else:

                    # Character fallback.
                    for ch in token:

                        ch_id = self.vocab.get(
                            ch
                        )

                        if ch_id is None:

                            ids.append(
                                self.unk_id
                            )

                        else:

                            ids.append(
                                ch_id
                            )

        return ids

    # ========================================================
    # DECODE
    # ========================================================

    def decode(
        self,
        ids,
    ):

        tokens = []

        for idx in ids:

            idx = int(idx)

            tokens.append(
                self.id_to_token.get(
                    idx,
                    self.unk_token,
                )
            )

        return "".join(
            tokens
        )

    # ========================================================
    # VOCAB SIZE
    # ========================================================

    def get_vocab_size(self):

        return self.vocab_size


# ============================================================
# LOAD TOKENIZER
# ============================================================

def load_tokenizer(
    tokenizer_dir,
):

    tokenizer_dir = str(
        tokenizer_dir
    )

    vocab_path = (
        f"{tokenizer_dir}/vocab.json"
    )

    merges_path = (
        f"{tokenizer_dir}/merges.txt"
    )

    config_path = (
        f"{tokenizer_dir}/tokenizer_config.json"
    )

    # --------------------------------------------------------
    # Vocabulary
    # --------------------------------------------------------

    with open(
        vocab_path,
        "r",
        encoding="utf-8",
    ) as f:

        vocab = json.load(f)

    # --------------------------------------------------------
    # JSONL merges
    # --------------------------------------------------------

    merges = []

    with open(
        merges_path,
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            line = line.rstrip(
                "\n"
            )

            if not line:
                continue

            pair = json.loads(
                line
            )

            if (
                not isinstance(pair, list)
                or len(pair) != 2
            ):

                raise ValueError(
                    "Invalid merge entry: "
                    f"{pair}"
                )

            merges.append(
                (
                    pair[0],
                    pair[1],
                )
            )

    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    with open(
        config_path,
        "r",
        encoding="utf-8",
    ) as f:

        config = json.load(f)

    # --------------------------------------------------------
    # Construct tokenizer
    # --------------------------------------------------------

    tokenizer = BPETokenizer(
        vocab=vocab,
        merges=merges,
        special_tokens=config[
            "special_tokens"
        ],
    )

    return tokenizer, config
'''


with open(
    PYTHON_PATH,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        tokenizer_code.strip()
        + "\n"
    )


# ============================================================
# 7. FILE HASHING
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# 8. SAVE MANIFEST
# ============================================================

manifest = {

    "tokenizer_type": "custom_bpe",

    "vocab_size": len(vocab),

    "num_merges": len(merges),

    "initial_vocab_size":
        initial_vocab_from_stats,

    "character_count":
        character_count,

    "special_token_count":
        special_count,

    "special_token_ids": {
        "pad": vocab[PAD_TOKEN],
        "unk": vocab[UNK_TOKEN],
        "bos": vocab[BOS_TOKEN],
        "eos": vocab[EOS_TOKEN],
    },

    "files": {
        "vocab": VOCAB_PATH.name,
        "merges": MERGES_PATH.name,
        "config": CONFIG_PATH.name,
        "python": PYTHON_PATH.name,
    },

    "merge_format":
        "json_lines",

    "dtype":
        "uint16",

    "sha256": {
        "vocab.json":
            sha256_file(VOCAB_PATH),

        "merges.txt":
            sha256_file(MERGES_PATH),

        "tokenizer_config.json":
            sha256_file(CONFIG_PATH),

        "tokenizer.py":
            sha256_file(PYTHON_PATH),
    },
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 9. LOAD SAVED TOKENIZER BACK
# ============================================================

spec = importlib.util.spec_from_file_location(
    "saved_tokenizer_module",
    PYTHON_PATH,
)

saved_tokenizer_module = (
    importlib.util.module_from_spec(
        spec
    )
)

spec.loader.exec_module(
    saved_tokenizer_module
)

(
    saved_tokenizer,
    saved_config,
) = saved_tokenizer_module.load_tokenizer(
    FINAL_TOKENIZER_DIR
)


# ============================================================
# 10. VERIFY SAVED TOKENIZER
# ============================================================

assert (
    saved_tokenizer.get_vocab_size()
    == VOCAB_SIZE
)

assert (
    len(saved_tokenizer.merges)
    == expected_merges
)

assert (
    saved_tokenizer.pad_id
    == vocab[PAD_TOKEN]
)

assert (
    saved_tokenizer.unk_id
    == vocab[UNK_TOKEN]
)

assert (
    saved_tokenizer.bos_id
    == vocab[BOS_TOKEN]
)

assert (
    saved_tokenizer.eos_id
    == vocab[EOS_TOKEN]
)


# ============================================================
# 11. ROUND-TRIP TEST
#
# Includes:
#   spaces
#   punctuation
#   newlines
#   Unicode
# ============================================================

test_text = (
    "The court held that the matter "
    "was dismissed.\n\n"
    "Section 12 applies.\n"
    "भारत न्यायालय"
)

test_ids = saved_tokenizer.encode(
    test_text
)

test_decoded = saved_tokenizer.decode(
    test_ids
)

assert (
    test_decoded
    == normalize_text(test_text)
)


# ============================================================
# 12. TOKEN-ID RANGE TEST
# ============================================================

assert all(
    0 <= int(token_id)
    < VOCAB_SIZE
    for token_id in test_ids
)


# ============================================================
# 13. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("FINAL TOKENIZER PACKAGE")
print("=" * 70)

print(
    "Vocabulary size :",
    saved_tokenizer.get_vocab_size(),
)

print(
    "Number of merges:",
    len(saved_tokenizer.merges),
)

print(
    "Initial vocab   :",
    initial_vocab_from_stats,
)

print(
    "Character count :",
    character_count,
)

print(
    "PAD ID          :",
    saved_tokenizer.pad_id,
)

print(
    "UNK ID          :",
    saved_tokenizer.unk_id,
)

print(
    "BOS ID          :",
    saved_tokenizer.bos_id,
)

print(
    "EOS ID          :",
    saved_tokenizer.eos_id,
)

print(
    "Test tokens     :",
    len(test_ids),
)

print(
    "Round-trip exact:",
    test_decoded
    == normalize_text(test_text),
)

print("\nSaved files:")

for path in sorted(
    FINAL_TOKENIZER_DIR.iterdir()
):

    print(
        f"  {path.name:25s}"
        f" {path.stat().st_size / 1024:.1f} KB"
    )


# ============================================================
# 14. FINAL ASSERTIONS
# ============================================================

assert (
    saved_tokenizer.get_vocab_size()
    == VOCAB_SIZE
)

assert (
    len(saved_tokenizer.merges)
    == expected_merges
)

assert (
    test_decoded
    == normalize_text(test_text)
)

print(
    "\n✓ TOKENIZER PACKAGE READY"
)

print(
    "Directory:",
    FINAL_TOKENIZER_DIR,
)

BPE STATE VALIDATION
Special tokens              : 4
Characters                  : 562
Initial vocab from stats    : 566
Initial vocab from BPE      : 566
BPE merges                  : 31438
Final vocabulary            : 32004

✓ BPE vocabulary and merge history are internally consistent.

FINAL TOKENIZER PACKAGE
Vocabulary size : 32004
Number of merges: 31438
Initial vocab   : 566
Character count : 562
PAD ID          : 0
UNK ID          : 1
BOS ID          : 2
EOS ID          : 3
Test tokens     : 39
Round-trip exact: True

Saved files:
  __pycache__               4.0 KB
  manifest.json             0.8 KB
  merges.txt                477.5 KB
  tokenizer.py              8.1 KB
  tokenizer_config.json     0.6 KB
  vocab.json                598.0 KB

✓ TOKENIZER PACKAGE READY
Directory: /kaggle/working/legal_tokenizer_v3/tokenizer/final_tokenizer
